# Tokenizer


## Step 1 : creating tokens


In [1]:
with open('the-verdict.txt','r',encoding='utf-8') as f:
    raw_text=f.read()

In [2]:
print('Total number of character :', len(raw_text))
print(raw_text[:99])

Total number of character : 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


Our goal is to tokenize this 20479 char short story into individual words and special characters that we cam then turn into embedding for llm training   

In [3]:
import re
text="hello, world. this, is the first."
result=re.split(r'(\s)',text) # split wherever there is a space
print(result)

['hello,', ' ', 'world.', ' ', 'this,', ' ', 'is', ' ', 'the', ' ', 'first.']


the result is  list of individual words, whitespaces, and punctuation characters.

modify the regular expression splits on whitespaces(\s) and commas , periods[,.]:

In [4]:
result=re.split(r'([,.]|\s)',text)
print(result)

['hello', ',', '', ' ', 'world', '.', '', ' ', 'this', ',', '', ' ', 'is', ' ', 'the', ' ', 'first', '.', '']


small issue - list still has whitespace characters. we can remove redundant characters safely:

In [5]:
result=[item for item in result if item.strip()] # remove empty strings
print(result)

['hello', ',', 'world', '.', 'this', ',', 'is', 'the', 'first', '.']


Removing whitespaces or not ; 
When developing a simple tokenizer, whether we should encode whitespaces as separate character or just remove them depend on our application and its requirements. removing whitespaces reduce memory usage and computation requirement. However keeping whitespaces can be useful if we train model that are sensitive to structure of text(python code sensitive to indentation and spacing). In this case we are going to remove the whitespaces.

The tokenization scheme that we had devised above works well on simple sample text. Modify it so that it also handle punctuation marks like question mark, quotation marks, and the double dashes etc. 

In [12]:
text="Hello, world. Is this-- a test?"
result=re.split(r'([,.:;?!_"()\']|--|\s)',text)
result=[item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


Now we got a basic tokenizer working, apply to edith wharton the verdict

In [14]:
preprocessed=re.split(r'([,.:;?!_"()\']|--|\s)',raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:100])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--']


In [15]:
print(len(preprocessed))

4690


# Step 2 -> Create Token IDs

Create a list of all unique tokens and sort them alphabetically to determine the vocab size

In [21]:
all_words=set(preprocessed)
all_words=sorted(all_words)
vocab_size=len(all_words)
# for i in range(vocab_size):
#     print(all_words[i])
print(vocab_size)

1130


Create the vocab

In [24]:
vocab={token:integer for integer,token in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
    print(item)
    if i>=100:
        break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)
('How', 52)
('I', 53)
('If', 54)
('In', 55)
('It', 56)
('Jack', 57)
('Jove', 58)
('Just', 59)
('Lord', 60)
('Made', 61)
('Miss', 62)
('Money', 63)
('Monte', 64)
('Moon-dancers', 65)
('Mr', 66)
('Mrs', 67)
('My', 68)
('Never', 69)
('No', 70)
('Now', 71)
('Nutley', 72)
('Of', 73)
('Oh', 74)
('On', 75)
('Once', 76)
('Only', 77)
('

Implement a complete tokenizer in Python.
Class will have an encode method that splits text into tokens and carries out the string to integer mapping 
In addition we will implement a decode method that carries out the reverse integer-to-string mapping to convert token id to word

In [29]:
class SimpleToeknizer:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.:;?!_"()\']|--|\s)',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        ids=[self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self,ids):
        text=" ".join([self.int_to_str[i] for i in ids])
        # replace the spaces before the specified punctuations
        text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [32]:
tokenizer=SimpleToeknizer(vocab)
text=""""It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)  
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [33]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

Now lets apply it to something not in vocab

In [34]:
text='Hello, do you like tea?'
print(tokenizer.encode(text))

KeyError: 'Hello'

# Adding special contxt tokens
In prev sections, we implemented a simpletokenizer and  applied it to a passage from training set.
Now, we will modify this tokenizer to handle unknown words 


In [36]:
all_tokens=sorted(set(preprocessed))
all_tokens.extend(['<|endoftext|>','<|unk|>'])
vocab={token:integer for integer,token in enumerate(all_tokens)}

In [37]:
len(vocab)

1132

In [40]:
#printing last 5 entries of updated vocab
for i,items in enumerate(list(vocab.items())[-5::]):
    print(items)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [41]:
class simpletokenizerv2:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}
    
    def encode(self,text):
        preprocessed=re.split(r'([,.:;?!_"()\']|--|\s)',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        preprocessed=[
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids=[self.str_to_int[item] for item in preprocessed]
        return ids

    def decode(self,ids):
        text=" ".join([self.int_to_str[i] for i in ids])
        # replace the spaces before the specified punctuations
        text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [43]:
tokenizer=simpletokenizerv2(vocab)
text1="Hello, do you like tea?"
text2="In the sunlit terraces of the palace."
text=" <|endoftext|> ".join([text1,text2])
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [44]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [45]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'